# Tests des fonctions de calcul de la saturation

In [1]:
import datetime

import numpy as np
import pandas as pd
from saturation_image_quali import (
    to_sampled_sessions,
    to_sampled_state_grp,
    to_sampled_state_poc,
    to_sampled_statuses,
    to_state_grp_d,
    to_state_grp_h,
)

from saturation import (
    hysteresis,
    # to_sampled_sessions,
    # to_sampled_state_grp,
    # to_sampled_state_pdc,
    # to_sampled_statuses,
)


## Test échantillonage des sessions

In [2]:
echantillons = 24
timestamp = pd.Timestamp('2025-04-25T00:00:00+02:00')
start = [1, 1.2, 3, 5.5, 9, 13.1, 20]
end = [2.1, 2.7, 5, 7.5, 12.1, 15.1, 22.6]

test = pd.DataFrame( {'start': [timestamp + pd.Timedelta(hours=val) for val in start],
                      'end': [timestamp + pd.Timedelta(hours=val) for val in end],
                      'id_pdc_itinerance': ['p1', 'p2', 'p2', 'p1', 'p2', 'p1', 'p2']})
pdc = test['id_pdc_itinerance'].unique()
init = pd.DataFrame( {'start': [timestamp + pd.Timedelta(hours=-1)] * len(pdc), 
                      'end': [timestamp + pd.Timedelta(hours=0.5)] * len(pdc),
                      'id_pdc_itinerance': pdc}) 
# p1 : [1, 2.1], [5.5, 7.5], [13.1, 15.1] -> [1.1, 2, 2]
# p2 : [1.2, 2.7], [3, 5], [9, 12.1], [20, 22.6] -> [1.5, 2, 3.1, 2.6]
sessions = to_sampled_sessions(test, init, timestamp, echantillons)

assert sessions.iloc[0]['occupation_pdc'] == 'occupe'
assert sessions.iloc[1]['occupation_pdc'] == 'occupe'
assert sessions.iloc[5]['occupation_pdc'] == 'f_libre'
assert sessions.iloc[6]['occupation_pdc'] == 'occupe'
assert sessions.iloc[34]['occupation_pdc'] == 'occupe'


In [3]:
sessions = to_sampled_sessions(test, init, timestamp, echantillons, min_duration=datetime.timedelta(hours=1.2) )

assert sessions.iloc[1]['occupation_pdc'] == 'f_libre'

# sessions

In [4]:
sessions = to_sampled_sessions(test, init, timestamp, echantillons, max_duration=datetime.timedelta(hours=3) )

assert sessions.iloc[34]['occupation_pdc'] == 'f_libre'

# sessions

In [5]:
echantillons = 24
timestamp = pd.Timestamp('2025-04-25T00:00:00+02:00')
start = [1, 1.2, 3, 5.5, 9, 13.1, 20]
end = [6.1, 2.7, 5, 7.5, 12.1, 15.1, 22.6]

test = pd.DataFrame( {'start': [timestamp + pd.Timedelta(hours=val) for val in start],
                      'end': [timestamp + pd.Timedelta(hours=val) for val in end],
                      'id_pdc_itinerance': ['p1', 'p2', 'p2', 'p1', 'p2', 'p1', 'p2']}) 
pdc = test['id_pdc_itinerance'].unique()
init = pd.DataFrame( {'start': [timestamp + pd.Timedelta(hours=-1)] * len(pdc), 
                      'end': [timestamp + pd.Timedelta(hours=0.5)] * len(pdc),
                      'id_pdc_itinerance': pdc}) 
# p1 : [1, 6.1], [5.5, 7.5], [13.1, 15.1]
# p2 : [1.2, 2.7], [3, 5], [9, 12.1], [20, 22.6]
sessions = to_sampled_sessions(test, init, timestamp, echantillons)

assert sessions.iloc[5]['occupation_pdc'] == 'occupe'
assert sessions.iloc[6]['occupation_pdc'] == 'occupe'

# sessions

## Test échantillonage des statuts

In [6]:
echantillons = 24
timestamp = pd.Timestamp('2025-04-25T00:00:00+02:00')
valeurs = [1, 1.2, 3, 3.5, 5, 6.1, 12] # p1 : [1, 3.5, 6.1] p2 : [1.2, 3, 5, 12]

test = pd.DataFrame( {'horodatage': [timestamp + pd.Timedelta(hours=val) for val in valeurs],
                      'etat_pdc':['en_service', 'hors_service', 'en_service', 'en_service', 
                                  'hors_service', 'en_service', 'hors_service'],
                      'id_pdc_itinerance': ['p1', 'p2', 'p2', 'p1', 'p2', 'p1', 'p2']})
pdc = test['id_pdc_itinerance'].unique()
init = pd.DataFrame( {'horodatage': [timestamp + pd.Timedelta(days=-1)] * len(pdc), 
                      'etat_pdc': ['en_service'] * len(pdc), 
                      'id_pdc_itinerance': pdc}) 
statuses = to_sampled_statuses(test, init, timestamp, echantillons)
assert len(statuses) == 48
assert statuses.iloc[25]['etat_pdc'] == 'en_service'
assert statuses.iloc[26]['etat_pdc'] == 'hors_service'
statuses = to_sampled_statuses(test, init, timestamp, echantillons, datetime.timedelta(hours=1.9))
assert len(statuses) == 48
assert statuses.iloc[25]['etat_pdc'] == 'en_service'
assert statuses.iloc[26]['etat_pdc'] == 'en_service'

# statuses

## Test assemblage des sessions et des statuts

In [7]:
sessions = pd.DataFrame({'id_pdc_itinerance': ['p1', 'p1', 'p1', 'p2', 'p2', 'p2', 'p3', 'p3', 'p3'], 
                       'periode': [0,1,2,0,1,2,0,1,2],
                       'occupation_pdc': ['occupe', 'f_libre', 'occupe', 'f_libre', 'occupe', 'f_libre','f_libre', 'occupe', 'f_libre']})
status = pd.DataFrame({'id_pdc_itinerance': ['p1', 'p1', 'p1', 'p3', 'p3', 'p3', 'p4', 'p4', 'p4'], 
                       'periode': [0,1,2,0,1,2, 0,1,2],
                       'etat_pdc': ['hors_service', 'hors_service', 'en_service', 'en_service', 'hors_service', 'hors_service', 'en_service', 'hors_service', 'en_service']})
merged = pd.merge(sessions, status, how='outer', on=['id_pdc_itinerance', 'periode']).fillna('aaa')
merged

,id_pdc_itinerance,periode,occupation_pdc,etat_pdc
0,p1,0,occupe,hors_service
1,p1,1,f_libre,hors_service
2,p1,2,occupe,en_service
3,p2,0,f_libre,aaa
4,p2,1,occupe,aaa
5,p2,2,f_libre,aaa
6,p3,0,f_libre,en_service
7,p3,1,occupe,hors_service
8,p3,2,f_libre,hors_service
9,p4,0,aaa,en_service


In [8]:
merged = to_sampled_state_poc(sessions, status)
assert list(merged['state'][0:4]) == ['occupe', 'hors_service', 'occupe', 'libre']
merged

,id_pdc_itinerance,periode,state
0,p1,0,occupe
1,p1,1,hors_service
2,p1,2,occupe
3,p2,0,libre
4,p2,1,occupe
5,p2,2,libre
6,p3,0,libre
7,p3,1,occupe
8,p3,2,hors_service
9,p4,0,libre


In [9]:
merged["occupe"] = merged["state"] == "occupe"
merged["hors_service"] = merged["state"] == "hors_service"
occupe_hs = merged['occupe'] | merged['hors_service']
merged['pleine_utilisation'] = occupe_hs | occupe_hs.shift(fill_value=False)
merged

,id_pdc_itinerance,periode,state,occupe,hors_service,pleine_utilisation
0,p1,0,occupe,True,False,True
1,p1,1,hors_service,False,True,True
2,p1,2,occupe,True,False,True
3,p2,0,libre,False,False,True
4,p2,1,occupe,True,False,True
5,p2,2,libre,False,False,True
6,p3,0,libre,False,False,False
7,p3,1,occupe,True,False,True
8,p3,2,hors_service,False,True,True
9,p4,0,libre,False,False,True


## Test état global échantillonné d'un groupement de pdc

In [10]:
test = pd.DataFrame({'id_pdc_itinerance': ['p1', 'p1', 'p1', 'p2', 'p2', 'p2', 'p3', 'p3', 'p3'],
                     'periode' : [0, 1, 2, 0, 1, 2, 0, 1, 2],
                     'state' : ['occupe', 'hors_service', 'occupe', 'libre', 'occupe', 'libre', 'libre', 'occupe', 'hors_service']})
stations = pd.DataFrame({'id_pdc_itinerance': ['p1', 'p2', 'p3'],
                         'id_station_itinerance': ['s1', 's1', 's2']}) 
to_sampled_state_grp(test, stations, 'id_station_itinerance', 0.1, 0.2, add_full_use=True)

,id_station_itinerance,periode,occupe,hors_service,libre,pleine_utilisation,nb_pdc,hs,inactif,sature,surcharge,actif,state,pu
0,s1,0,1,0,1,2,2,False,False,False,False,True,3,True
1,s1,1,1,1,0,2,2,False,False,True,False,False,5,True
2,s1,2,1,0,1,2,2,False,False,False,False,True,3,True
3,s2,0,0,0,1,0,1,False,True,False,False,False,2,False
4,s2,1,1,0,0,1,1,False,False,True,False,False,5,True
5,s2,2,0,1,0,1,1,True,False,False,False,False,1,True


## Test de la pleine utilisation

In [11]:
test = pd.DataFrame({'id_pdc_itinerance': ['p1', 'p1', 'p1', 'p1',
                                           'p2', 'p2', 'p2', 'p2',
                                           'p3', 'p3', 'p3', 'p3',
                                           'p4', 'p4', 'p4', 'p4',
                                           'p5', 'p5', 'p5', 'p5',
                                           'p6', 'p6', 'p6', 'p3'],
                     'periode' : [timestamp + pd.Timedelta(hours=val) for val in [0, 1, 2, 3] * 6],
                     # 'periode' : timestamp,
                     'state' : ['libre', 'occupe', 'occupe', 'occupe'] * 6})
stations = pd.DataFrame({'id_pdc_itinerance': ['p1', 'p2', 'p3', 'p4', 'p5', 'p6'],
                         'id_station_itinerance': ['s1'] *6}) 
res = to_sampled_state_grp(test, stations, 'id_station_itinerance', 0.01, 0.2)
assert res['sature'][3]
res = to_sampled_state_grp(test, stations, 'id_station_itinerance', 0.01, 0.2, add_full_use=True)
assert (res['sature'] == res['pu']).all()
test.loc[2,'state'] = 'libre'
test.loc[3,'state'] = 'libre'
res = to_sampled_state_grp(test, stations, 'id_station_itinerance', 0.01, 0.2, add_full_use=True)
assert not res['sature'][2]
assert res['pu'][2]
assert res['pu'][3]
test.loc[6,'state'] = 'libre'
test.loc[7,'state'] = 'libre'
res = to_sampled_state_grp(test, stations, 'id_station_itinerance', 0.01, 0.2, add_full_use=True)
assert res['pu'][2]
assert not res['pu'][3]
res

,id_station_itinerance,periode,occupe,hors_service,libre,pleine_utilisation,nb_pdc,hs,inactif,sature,surcharge,actif,state,pu
0,s1,2025-04-25 00:00:00+02:00,0,0,6,3,6,False,True,False,False,False,2,False
1,s1,2025-04-25 01:00:00+02:00,6,0,0,6,6,False,False,True,False,False,5,True
2,s1,2025-04-25 02:00:00+02:00,4,0,2,6,6,False,False,False,False,True,3,True
3,s1,2025-04-25 03:00:00+02:00,4,0,2,4,6,False,False,False,False,True,3,False


In [12]:
res_h = to_state_grp_h(res, 'id_station_itinerance', 24, 45 )

TypeError: datetime64 type does not support sum operations

In [ ]:
'state' in test.columns

In [ ]:
test = pd.DataFrame({'id_pdc_itinerance': ['p1', 'p1', 'p1', 'p1', 'p1', 'p1',
                                           'p2', 'p2', 'p2', 'p2', 'p2', 'p2'],
                     'periode' : [0, 1, 2, 3, 4, 5,
                                  0, 1, 2, 3, 4, 5],
                     'state' : ['occupe', 'hors_service', 'occupe', 'occupe', 'hors_service', 'libre',
                                'libre', 'libre', 'occupe', 'hors_service', 'hors_service', 'libre']})
stations = pd.DataFrame({'id_pdc_itinerance': ['p1', 'p2'],
                         'id_station_itinerance': ['s1', 's1']}) 
to_sampled_state_grp(test, stations, 'id_station_itinerance', 0.1, 0.2)

In [ ]:
test = pd.DataFrame({'name':         ['hs', 'inactif', 'sature', 'surcharge', 'actif'],
                     'occupe':       [0, 0, 5, 3, 2],
                     'hors_service': [6, 2, 1, 2, 2],
                     'libre':        [0, 4, 0, 1, 2],
                     'nb_pdc':       [6, 6, 6, 6, 6]})
# 2e partie de la fonction : to_sampled_state_grp
test['hs'] = (test['libre'] + test['occupe'] == 0) & (test['hors_service'] > 0)
test['inactif'] = ~test['hs'] & (test['occupe'] == 0)
test['sature'] = ~test['hs'] & ~test['inactif'] & (test['libre']/test['nb_pdc'] < 0.1)
test['surcharge'] = ~test['hs'] & ~test['inactif'] & ~test['sature'] & (test['libre']/test['nb_pdc'] < 0.2)
test['actif'] = ~test['hs'] & ~test['inactif'] & ~test['sature'] & ~test['surcharge']
test['state'] = test['hs'] + test['inactif'] * 2 + test['actif'] * 3 + test['surcharge'] * 4 + test['sature'] * 5
test

## test de l'hysteresis

In [ ]:
# seuil à 6 et 9
serie = pd.Series([1, 2, 5, 7, 5, 8, 10, 12, 8, 11, 8, 5, 7, 2])
res = hysteresis(serie, 6, 9)
assert res[6:10].all() == True
assert res[:5].all() == False

In [ ]:
test = pd.DataFrame({'id_pdc_itinerance': ['pdc1'] * 10 + ['pdc2'] * 10,
                     'periode':      [0, 1, 2, 3, 4, 5, 6, 7, 8, 9] * 2,
                     'occupe':       [0, 1, 1, 0, 0, 1, 0, 0, 0, 1] * 2,
                     'hors_service': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0] * 2, 
                     'libre':        [1, 0, 0, 1, 1, 0, 1, 1, 1, 0] * 2,
                     'nb_pdc':       [1, 1, 1, 1, 1, 1, 1, 1, 1, 1] * 2})
hyst = 3
pleine_occupation = test['occupe'].copy()
for i in range(1, hyst):
    pleine_occupation += [0] * i + list(test['occupe'])[0:len(test) - i]
f_id_pdc_itinerance = pd.Series(['aucun'] * hyst + list(test['id_pdc_itinerance'])[0:len(test) - hyst])
test['valid_po'] = f_id_pdc_itinerance == test['id_pdc_itinerance']
test['po'] = pleine_occupation > 0
test

In [ ]:
def maxi_periode_PU(state_grp: pd.DataFrame, group_name: str) -> pd.DataFrame:
    '''Calculate de PU periode with the maximum duration.'''
    df = state_grp.sort_values(by=[group_name, 'periode']).reset_index(drop=True)

    groups = df["occupe"].ne(
        df.groupby(group_name)["occupe"].shift()
    ).groupby(df[group_name]).cumsum()

    valid_groups = df[df["occupe"]].assign(valid_group=groups[df["occupe"]])

    maxi_period = valid_groups.groupby([group_name, "valid_group"]).agg(
            start=("periode", "first"),
            end=("periode", "last"),
        )
    maxi_period['duration'] = maxi_period['end'] - maxi_period['start']
    maxi_period = maxi_period.loc[
        maxi_period.groupby(level=0)["duration"].idxmax()
    ]
    return maxi_period

timestamp = pd.Timestamp('2025-04-25T00:00:00+02:00')
periodes = [1, 2, 3, 4, 5, 6, 7, 8, 9] * 2

df = pd.DataFrame({'occupe': [False, True, True, False, True, True, True, False, True,
                              False, True, True, True, True, False, True, False, True],
                   'station': ['s1'] * 9 + ['s2'] * 9,
                   'periode': [timestamp + pd.Timedelta(hours=per) for per in periodes]})
pu_duration = maxi_periode_PU(df, 'station')
assert pu_duration['duration'].iloc[0] == datetime.timedelta(hours=2)
assert pu_duration['duration'].iloc[1] == datetime.timedelta(hours=3)


In [ ]:
latency = datetime.timedelta(minutes=5)
if latency :
    df = df.assign(periode= df['periode']+latency)
else :
    print('prout')
df

In [ ]:
grp, masque

In [ ]:
masque[masque]

In [ ]:
longueur_max = s.groupby(grp).sum().max()
longueur_max

In [ ]:
tailles = s.groupby(grp).sum()
tailles

In [ ]:
idx = tailles.idxmax()
idx, tailles[idx]

In [ ]:
masque = grp==idx
masque

In [ ]:
debut = masque.idxmax()
fin = debut + tailles[idx] - 1
debut, fin

In [ ]:

def decal(data, timestamp, samples_per_day):
    samples = pd.date_range(
        start=timestamp,
        end=timestamp + pd.Timedelta(days=1),
        periods=samples_per_day + 1,
    )
    state = data.sort_values(
        by=["id_pdc_itinerance", "horodatage"]
    )
    state = state[(state["etat_pdc"] != "inconnu")]
    state["f_horodatage"] = list(state["horodatage"])[1 : len(state)] + [
        samples[samples_per_day]
    ]
    state["f_id_pdc_itinerance"] = list(state["id_pdc_itinerance"])[1 : len(state)] + [
        "aucun"
    ]
    return state

def decal2(data, timestamp, samples_per_day):
    samples = pd.date_range(
        start=timestamp,
        end=timestamp + pd.Timedelta(days=1),
        periods=samples_per_day + 1,
    )
    state = data.sort_values(
        by=["id_pdc_itinerance", "horodatage"]
    )
    state = state[(state["etat_pdc"] != "inconnu")]
    state["f_horodatage"] = state["horodatage"].shift(-1)
    state.loc[state.index[-1], "f_horodatage"] = samples[samples_per_day]
    state["f_id_pdc_itinerance"] = state["id_pdc_itinerance"].shift(-1)
    state.loc[state.index[-1], "f_id_pdc_itinerance"] = "aucun"
    return state

echantillons = 24
timestamp = pd.Timestamp('2025-04-25T00:00:00+02:00')
valeurs = [1, 1.2, 3, 3.5, 5, 6.1, 12] # p1 : [1, 3.5, 6.1] p2 : [1.2, 3, 5, 12]

test = pd.DataFrame( {'horodatage': [timestamp + pd.Timedelta(hours=val) for val in valeurs],
                      'etat_pdc':['en_service', 'hors_service', 'en_service', 'en_service', 
                                  'hors_service', 'en_service', 'hors_service'],
                      'id_pdc_itinerance': ['p1', 'p2', 'p2', 'p1', 'p2', 'p1', 'p2']})

res = decal(test, timestamp, echantillons)
res

In [ ]:
res2 = decal2(test, timestamp, echantillons)
res2